보통 언어에서 많이 사용하는 `getter`, `setter` 방식이다.

In [ ]:
class OldResistor:
    def __init__(self, ohms):
        self._ohms = ohms
    
    def get_ohms(self):
        return self._ohms
    
    def set_ohms(self, ohms):
        self._ohms = ohms
        
r0 = OldResistor(50e3)
print('이전: ', r0.get_ohms())
r0.set_ohms(10e3)
print('이후: ', r0.get_ohms())

필드 값 증가시키는 등 연산에서는 이런 메소드로 인해 코드가 지저분해진다.

In [ ]:
r0.set_ohms(r0.get_ohms() - 4e3)
assert r0.get_ohms() == 6e3

이런 유틸리티 메서드를 사용하면 기능을 캡슐화하고, 필드 사용을 검증하고, 경계를 설정하기 쉬워진다. 하지만 파이썬에서는 명시적인 `getter`, `setter` 메서드 구현할 필요가 전혀 없다.

In [ ]:
class Resistor:
    def __init__(self, ohms):
        self.ohms = ohms
        self.voltage = 0
        self.current = 0

r1 = Resistor(50e3)
print(r1.ohms)
r1.ohms = 10e3
print(r1.ohms)

이렇게 애트리뷰트를 사용하면 필드를 제자리에서 증가시키는 등의 연산이 더 자연스럽고 명확해진다.

In [ ]:
r1.ohms += 5e3
print(r1.ohms)

애트리뷰트 설정할 때 특별한 기능을 수행해야 한다면 `@property` 데코레이터(Better way 26)와 대응하는 `setter` 애트리뷰트로 옮겨갈 수 있다.

In [ ]:
class VoltageResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)
        self._voltage = 0

    @property
    def voltage(self):
        return self._voltage

    @voltage.setter
    def voltage(self, voltage):
        self._voltage = voltage
        self.current = self._voltage / self.ohms

이제 `voltage` 프로퍼티에 대입하면 `@voltage.setter` 메서드가 호출된다.

In [ ]:
r2 = VoltageResistance(1e3)
print(f'이전: {r2.current:.2f} 암페어')
r2.voltage = 10
print(f'이후: {r2.current:.2f} 암페어')

프로퍼티 `setter` 메서드를 이용해서 전달값 검증을 수행할 수 있다.

In [ ]:
class BoundedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)

    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if ohms <= 0:
            raise ValueError(f'저항 > 0이어야 합니다. 실제 값: {ohms}')
        self._ohms = ohms

잘못된 저항값을 대입하면 예외가 발생한다.

In [ ]:
r3 = BoundedResistance(1e3)
r3.ohms = 0

생성자에 잘못된 값을 넘기는 경우에도 마찬가지로 예외가 발생한다.

In [ ]:
BoundedResistance(-5)

예외가 발생하는 이유
1. `BoundedResistance(-5)` 객체 생성
1. `BoundedResistance.__init__` 생성자 호출
1. `Registor.__init__` 생성자 호출
1. `self.ohms = -5` 대입문 실행
1. `@ohms.setter` 메서드 호출 _(오류)_

심지어 `@property`를 사용해 부모 클래스에 정의된 애트리뷰트를 불변으로 만들 수도 있다.

In [ ]:
class FixedResistance(Resistor):
    def __init__(self, ohms):
        super().__init__(ohms)

    @property
    def ohms(self):
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        if hasattr(self, '_ohms'):
            raise AttributeError("Ohms는 불변객체입니다")
        self._ohms = ohms

이 객체를 만들고 프로퍼티에 값을 대입하면 예외가 발생한다.

In [ ]:
r4 = FixedResistance(1e3)
r4.ohms = 2e3

`@property` 이용해서 `getter`/`setter` 구현할 때는 예기치 않은 동작을 수행하지 않도록 주의해야 한다. 예를 들어 `getter` 프로퍼티 메서드 안에서 다른 애트리뷰트를 설정하면 안된다.

In [ ]:
class MysteriousResistor(Resistor):
    @property
    def ohms(self):
        self.voltage = self._ohms * self.current
        return self._ohms

    @ohms.setter
    def ohms(self, ohms):
        self._ohms = ohms

`ohms` 값을 조회하는데 `voltage` 값이 변경되어 출력된다.

In [ ]:
r7 = MysteriousResistor(10)
r7.current = 0.01
print(f'이전: {r7.voltage:.2f}')
r7.ohms
print(f'이후: {r7.voltage:.2f}')

가장 좋은 정책은 관련이 있는 객체 상태를 `@property.setter` 메서드 안에서만 변경 하는 것이다.

이런건 하지 말자.
- 동적 모듈 임포트
- 아주 시간이 오래 걸리는 도우미 함수를 호출
- I/O를 수행
- 비용이 매우 많이 드는 DB 쿼리 등

`@property` 가장 큰 단점은 애트리뷰트를 처리하는 메서드가 하위 클래스 사이에서만 공유될 수 있다는 점이다. 하지만 파이썬은 재사용 가능한 프로퍼티 로직을 구현할 때는 물론 다른 여러 용도에도 사용할 수 있는 _디스크립터(descriptor)_ 를 제공한다. (Better way 46)